# step1 — 준수율 절벽 (RQ1)

**무엇을 확인하나:** 문맥(함수 12개)에 **지침을 어긴 이름이 많아질수록**, 모델이 새로 쓰는 함수 이름의
**준수율이 떨어지는가**, 어디서 급락(절벽)하는가. 실패가 실재함을 보이는 스텝(뒤 스텝들의 전제).

**어떻게:** 지침 방향 **2종**(‘camelCase 써라’ / ‘snake_case 써라’)으로 각각. 문맥의 **위반 개수를 0~12 전부**
바꿔가며, 모델이 실제로 **생성한 첫 함수 이름**이 지침 표기와 맞는지 본다. (언어는 파이썬 고정)

**고정 설정:** 4모델 · 504이름(42묶음) · 무작위값 42 · 그리디 생성 1턴 · 파이썬.

> **방향 2종을 쓰는 이유:** step4·5와 **일관**되게(대칭 확인). ‘camel만 특별한가?’를 막으려고 양방향을 본다.
> 파이썬 관용이 snake라, ‘snake 써라’ 쪽 기본 준수율이 더 높고 ‘camel 써라’가 더 어려울 수 있음(대조점).

**모델 하나씩.** 셀 ③ `PICK` → 셀 ④~⑦. 끝나면 `PICK` 바꿔 반복. 끊겨도 저장된 건 건너뜀.


In [ ]:
# ① 환경 설정 — 설치, GPU 확인, 무작위값 42 고정

import random, numpy as np, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (매우 느림)')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('무작위값 고정:', SEED)


In [ ]:
# ② 저장소 클론 및 브랜치 체크아웃
import os
if not os.path.isdir('HCLT_2026'):
    !git clone https://github.com/deanjs/HCLT_2026.git
%cd HCLT_2026
BRANCH = 'main'
!git fetch --quiet origin $BRANCH
!git checkout $BRANCH
!git pull --quiet origin $BRANCH
!pip install -q -r requirements.txt   # 저장소를 받은 뒤라야 이 파일이 있다
!pip install -e . -q
import sys; sys.path.insert(0, 'src')

# ── 버전 확인 ─────────────────────────────────────────────────────────────
# 이 실험은 transformers 의 KV 캐시 자료구조를 **직접 편집**한다. 버전이 다르면
# 개입이 일어나는 대상 자체가 달라져 기존 결과와 비교할 수 없다. 어긋나면 멈춘다.
import torch, transformers
_want = {}
for _line in open('requirements.txt', encoding='utf-8'):
    _line = _line.split('#')[0].strip()
    if '==' in _line:
        _k, _v = _line.split('==')
        _want[_k.strip()] = _v.strip()
_have = {'torch': torch.__version__.split('+')[0],
         'transformers': transformers.__version__}
print()
print(f"{'라이브러리':<16}{'요구':<14}{'지금':<14}")
_bad = []
for _k in ('torch', 'transformers'):
    _ok = _have[_k] == _want.get(_k)
    print(f"{_k:<16}{_want.get(_k, '-'):<14}{_have[_k]:<14}{'' if _ok else '  <- 다름'}")
    if not _ok:
        _bad.append(_k)
if _bad:
    raise RuntimeError(
        '라이브러리 버전이 요구와 다릅니다: ' + ', '.join(_bad) + '\n'
        '이 실험은 KV 캐시 자료구조를 직접 편집하므로, 버전이 다르면 개입 대상 자체가\n'
        '달라져 기존 결과와 비교할 수 없습니다.\n'
        '\n해결: 위 설치가 새 버전을 받았다면 **런타임 > 세션 다시 시작** 후\n'
        '셀 하나부터 다시 실행하십시오. 설치는 재시작해야 반영됩니다.\n'
        '\n실제 실행 환경은 결과 JSON 의 meta 에 자동 기록됩니다.')
print('버전 일치 - 진행합니다')


In [ ]:
# ③ 조건 설정 — 4모델 중 하나 골라 위반개수 0~12 x 지침 방향 2종 (파이썬)
from harness.conditions import (Condition, ModelSpec, PrecedingCode, Instruction,
                                Composition, InstructionForm, Notation)

MODELS = [
    ModelSpec(name='Qwen/Qwen2.5-Coder-3B-Instruct',           family='qwen',      dtype='float16'),
    ModelSpec(name='deepseek-ai/deepseek-coder-6.7b-instruct',  family='deepseek',  dtype='float16'),
    ModelSpec(name='unsloth/Llama-3.2-3B-Instruct',             family='llama',     dtype='float16'),
    ModelSpec(name='stabilityai/stable-code-instruct-3b',       family='stability', dtype='float16'),
]

# ★ 이번에 돌릴 모델 하나 (0=qwen, 1=deepseek, 2=llama, 3=stable)
PICK = 0
MODEL = MODELS[PICK]
# deepseek-6.7b OOM나면 아래 해제(8bit):
# if MODEL.family=='deepseek': MODEL = ModelSpec(name=MODEL.name, family='deepseek', dtype='float16', quantization='8bit')

DIRECTIONS = [Notation.CAMEL, Notation.SNAKE]   # 지침이 camel / snake
NCOMP  = list(range(13))            # 준수 개수 0~12 (= 위반 12~0)
BLOCKS = list(range(42))

def cond(target, ncomp, block):
    return Condition(model=MODEL,
        preceding=PrecedingCode(n_compliant=ncomp, n_functions=12, composition=Composition.POOL,
                                pool_block=block),   # lang 없음 = 파이썬
        instruction=Instruction(form=InstructionForm.POSITIVE, target_notation=target),
        seed=42)

conditions = [cond(t, nc, b) for t in DIRECTIONS for nc in NCOMP for b in BLOCKS]
print('모델:', MODEL.family, '| 조건 수:', len(conditions), '(=방향2 x 13점 x 42묶음)')
print('예(camel지침 위반6):', cond(Notation.CAMEL, 6, 0).slug())
print('예(snake지침 위반6):', cond(Notation.SNAKE, 6, 0).slug())


In [ ]:
# ③-2 분할 — 이 런타임이 맡을 몫. DeepSeek만 2분할, 나머지는 통으로.
# 한 칸씩 건너뛰며 나눈다 → 각 런타임이 위반 0~12개와 두 방향을 골고루 맡는다.
if MODEL.family == 'deepseek':
    SHARD, N_SHARDS = 0, 2        # ★ 런타임 A는 0, 런타임 B는 1
else:
    SHARD, N_SHARDS = 0, 1        # stable / llama / qwen 은 한 런타임에서 전부

conditions = conditions[SHARD::N_SHARDS]
print(f'{MODEL.family} | 이 런타임 몫 {len(conditions)}개 (전체 1092 중 {SHARD+1}/{N_SHARDS})')


In [ ]:
# ③-3 미리보기 — 모델 로딩 + 10개만 생성해 눈으로 확인. 결과를 저장하지 않는다.
#
# 왜 보는가: 자유 생성은 모델이 설명 문장과 기존 모듈 재출력부터 시작해 토큰 한도 안에
# 새 함수 이름에 닿지 못할 수 있다. 그러면 이름 추출기가 재출력된 선행 함수 이름을 집는다
# (DeepSeek·StableCode에서 실제로 발생). 접두 강제는 답변 자리에 "def "를 미리 채워
# 다음 토큰이 곧 함수 이름이 되게 한다. 두 방식을 나란히 찍으니 눈으로 비교하고 정하면 된다.
import time
from harness import run
from harness.model import load_model
from harness.conditions import Notation

handle = load_model(MODEL)
print(f'{MODEL.family} | 층수 {handle.num_layers} | 로드 완료\n')

PREVIEW_NCOMP = [12, 11, 10, 9, 8, 7, 6, 4, 2, 0]   # 준수 개수 → 위반 0,1,2,3,4,5,6,8,10,12

for nc in PREVIEW_NCOMP:
    c = cond(Notation.CAMEL, nc, 0)
    t0 = time.time(); free = run(c, handle=handle, max_turns=1, max_new_tokens=100)
    t_free = time.time() - t0
    t0 = time.time(); pre = run(c, handle=handle, max_turns=1, max_new_tokens=20,
                                forced_prefix='def')
    t_pre = time.time() - t0
    ef, ep = free.metrics.extra, pre.metrics.extra
    print('-' * 78)
    print(f'위반 {12 - nc:>2}개')
    print(f"  자유생성   {str(ef['turn_names'][0]):<32} {ef['turn_notations'][0]:<6} {t_free:5.1f}초")
    print(f"      원문 > {ef['turn_texts'][0][:150].strip()!r}")
    print(f"  접두강제   {str(ep['turn_names'][0]):<32} {ep['turn_notations'][0]:<6} {t_pre:5.1f}초")
    print(f"      원문 > {ep['turn_texts'][0][:80].strip()!r}")

n = len(conditions)
print('\n' + '=' * 78)
print(f'이 런타임 몫 {n}개 예상 시간 — 자유생성 {t_free * n / 60:.0f}분 / 접두강제 {t_pre * n / 60:.0f}분')
print('접두강제 쪽 이름이 과제(removeDuplicates / remove_duplicates)와 맞는지 확인하고 다음 셀로.')


In [ ]:
# ④ 실행 — 조건마다 즉시 저장(재개). 첫 함수 이름의 표기 준수 여부를 측정한다.
#
# METHOD: 미리보기(③-3)를 보고 고른다.
#   접두강제 = 답변 자리에 "def "를 채워 다음 토큰이 곧 함수 이름 (에코 원천 차단, 빠름)
#   자유생성 = 종전 방식. 기존 Qwen·StableCode 결과가 이 방식으로 측정됐다
METHOD = dict(max_new_tokens=20, forced_prefix='def')
# METHOD = dict(max_new_tokens=100)          # 자유 생성으로 되돌릴 때

from harness import ResultRecord, save_result, result_path

STEP = 'step1_cliff'
todo = [c for c in conditions if not result_path(c, step=STEP).exists()]
print(f'[{MODEL.family}] 이 런타임 몫 {len(conditions)} / 남은 {len(todo)}')

if todo:
    for i, c in enumerate(todo, 1):
        out = run(c, handle=handle, max_turns=1, **METHOD)
        save_result(ResultRecord(condition=out.condition, metrics=out.metrics,
                                 step=STEP, rq='RQ1'))
        if i % 50 == 0 or i == len(todo):
            e = out.metrics.extra
            nv = c.preceding.n_functions - c.preceding.n_compliant
            tgt = c.instruction.target_notation.value
            ok = '준수' if e['first_compliant'] else '위반'
            print(f"    [{i}/{len(todo)}] {tgt}지침 위반{nv}개 -> '{e['turn_names'][0]}' ({ok})")
    print('  완료.')
else:
    print('  이미 다 됨')


In [ ]:
# ⑤ 결과 로드 (이 모델 것)
from harness import result_path
from harness.results import load_result
recs = [load_result(result_path(c, step='step1_cliff')) for c in conditions
        if result_path(c, step='step1_cliff').exists()]
print('불러온 조건:', len(recs), '-> results/step1_cliff/')


In [ ]:
# ⑥ 요약 — 위반개수 vs 준수율 (지침 방향별, 절벽 곡선) + 신뢰구간 + 생성 예시 + 그림
import numpy as np, pandas as pd
from collections import defaultdict
import matplotlib.pyplot as plt

# (지침방향, 위반개수) -> [준수 0/1]
agg = defaultdict(lambda: defaultdict(list))
parsefail = defaultdict(lambda: defaultdict(int))
for r in recs:
    tgt = r.condition.instruction.target_notation.value
    nviol = r.condition.preceding.n_functions - r.condition.preceding.n_compliant
    ex = r.metrics.extra
    agg[tgt][nviol].append(1 if ex['first_compliant'] else 0)
    if ex['turn_names'][0] is None: parsefail[tgt][nviol] += 1

def rate_ci(vals):
    a = np.array(vals); p = a.mean(); n = len(a)
    ci = 1.96*np.sqrt(p*(1-p)/n) if n>0 else 0
    return p, ci

print('=== 지침 방향별 위반개수 vs 준수율 ===')
for tgt in ['camel','snake']:
    if tgt not in agg: continue
    xs = sorted(agg[tgt]); ps=[]
    for x in xs:
        p,ci = rate_ci(agg[tgt][x]); ps.append(p)
        print(f'  {tgt:6s}지침 위반 {x:2d}개: 준수율 {p*100:5.1f}% (±{ci*100:.1f})  이름못뽑음 {parsefail[tgt][x]}')
    print(f'    -> {tgt}: 위반0 {ps[0]*100:.0f}% ... 위반12 {ps[-1]*100:.0f}%')

# 절벽 그림 (영문 라벨)
fig, ax = plt.subplots(figsize=(7.5,4.5))
NAMEEN={'camel':'camelCase instruction','snake':'snake_case instruction'}
for tgt,color in [('camel','#2563eb'),('snake','#ea580c')]:
    if tgt not in agg: continue
    xs = sorted(agg[tgt]); P=[]; C=[]
    for x in xs:
        p,ci = rate_ci(agg[tgt][x]); P.append(p); C.append(ci)
    P=np.array(P); C=np.array(C)
    ax.plot(xs, P, marker='o', color=color, lw=2, label=NAMEEN[tgt])
    ax.fill_between(xs, P-C, P+C, color=color, alpha=.18)
ax.set_xlabel('Number of violation names in context (0-12)')
ax.set_ylabel('Compliance rate (first generated name)')
ax.set_title(f'{MODEL.family} — compliance cliff (step1, RQ1)')
ax.set_ylim(-0.02,1.02); ax.grid(alpha=.25); ax.legend(); plt.tight_layout(); plt.show()

# 생성 예시 몇 개 (눈으로 검증)
print('\n=== 생성 예시 (camel지침, 위반12개) ===')
shown=0
for r in recs:
    if shown>=3: break
    if r.condition.instruction.target_notation.value=='camel' and r.condition.preceding.n_compliant==0:
        t=r.metrics.extra['turn_texts'][0].replace(chr(10),' ')[:90]
        print(f"  이름 '{r.metrics.extra['turn_names'][0]}' ({r.metrics.extra['turn_notations'][0]}): {t}")
        shown+=1


In [ ]:
STEPS = ['step1_cliff']
# ⑧ 결과 zip으로 묶어 내려받기
import shutil, os, glob

def pack(step):
    d = f'results/{step}'
    if not os.path.isdir(d):
        print(f'  [건너뜀] {d} 폴더가 없다 — 이 스텝은 아직 안 돌렸다')
        return None
    n = len(glob.glob(f'{d}/*.json'))
    if n == 0:
        print(f'  [건너뜀] {d} 가 비어 있다')
        return None
    path = shutil.make_archive(step, 'zip', d)
    print(f'  {step}: {n}개 → {path} ({os.path.getsize(path)/1e6:.1f}MB)')
    return path

print('results/ 안에 있는 폴더:', sorted(os.listdir('results')) if os.path.isdir('results') else '(results 폴더 없음)')
print()
made = [p for p in (pack(s) for s in STEPS) if p]

if not made:
    print('\n내려받을 것이 없다. 실행 셀을 먼저 돌렸는지 확인할 것.')
else:
    try:
        from google.colab import files
        for p in made:
            files.download(p)
        print('\n다운로드 시작. 브라우저가 막으면 왼쪽 **파일 탐색기**에서 직접 받으면 된다.')
    except Exception as e:
        print(f'\nColab 자동 다운로드 불가({type(e).__name__}). 위 경로에서 직접 받을 것.')
